# Preprocess TCGA mutational profiles & Repair deficiency

In [1]:
# snmf_env
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
from matplotlib.colors import LogNorm
from matplotlib import cm
import os
import pickle as pkl
import umap

from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn import metrics

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

from SigProfilerAssignment import Analyzer as Analyze

%matplotlib inline
sns.set_theme(style="whitegrid")
sns.set(rc={'figure.figsize':(10.0,8.27)})

from SigProfilerMatrixGenerator import install as genInstall
genInstall.install('GRCh37')


# LOAD Bootstrapped functions
import sys
import os
# append project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

import importlib
from src.processing import bootstrapping as boot
# Reload the module to ensure we have the latest version
importlib.reload(boot)


/Users/sande/miniconda3/envs/snmf_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Tool       | Installed 
-----------------------
curl       | True      
wget       | False     
rsync      | True      


INFO - GRCh37 is already installed.


All reference files have been created.
To proceed with matrix_generation, please provide the path to your vcf files and an appropriate output path.
Installation complete.


<module 'src.processing.bootstrapping' from '/Users/sande/Documents/GitHub/SNMF/src/processing/bootstrapping.py'>

## Load data

In [2]:
project_root = os.path.join(os.getcwd(), "../..")




tcga_labels = tcga_labels.reset_index()
tcga_labels.index = tcga_labels['Sample'].str[:12]

tcga_labels

tcga_labels_HR = tcga_labels[tcga_labels['Cancer'] == cancer]
tcga_labels_HR

NameError: name 'tcga_labels' is not defined

In [5]:
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

cancer = 'BRCA'  # Example cancer type, change as needed


path_tcga_labels = os.path.join(project_root, "data", "processed", "TCGA", "annot.txt")
tcga_labels = pd.read_csv(path_tcga_labels, sep='\t', header=0, index_col=0)


tcga_count = pd.read_pickle(os.path.join(project_root, 'data', 'processed', 'TCGA','profiles','count', '{}.pkl'.format(cancer)))
tcga_profiles = pd.read_csv(os.path.join(project_root, 'data', 'processed', 'TCGA','profiles','norm_SBS', '{}.text'.format(cancer)), sep = '\t', index_col=0)

print(tcga_count.iloc[:10, ]['count_SBS'])
print(tcga_count.shape)

# TCGA_profiles
tcga_profiles

# Load 
# /Users/sande/Documents/GitHub/SNMF/data/processed/TCGA/volkova_tcga.xlsx

TCGA-3C-AAAU-01      46.0
TCGA-3C-AALI-01     808.0
TCGA-3C-AALJ-01      66.0
TCGA-3C-AALK-01      64.0
TCGA-4H-AAAK-01      33.0
TCGA-5L-AAT0-01     279.0
TCGA-5L-AAT1-01    1986.0
TCGA-5T-A9QA-01      53.0
TCGA-A1-A0SB-01      16.0
TCGA-A1-A0SD-01      40.0
Name: count_SBS, dtype: float64
(791, 261)


,TCGA-3C-AAAU-01,TCGA-3C-AALI-01,TCGA-3C-AALJ-01,TCGA-3C-AALK-01,TCGA-4H-AAAK-01,TCGA-5L-AAT0-01,TCGA-5L-AAT1-01,TCGA-5T-A9QA-01,TCGA-A1-A0SB-01,TCGA-A1-A0SD-01,...,TCGA-UL-AAZ6-01,TCGA-UU-A93S-01,TCGA-V7-A7HQ-01,TCGA-W8-A86G-01,TCGA-WT-AB41-01,TCGA-WT-AB44-01,TCGA-XX-A899-01,TCGA-XX-A89A-01,TCGA-Z7-A8R5-01,TCGA-Z7-A8R6-01
A[C>A]A,0.000000,0.003713,0.000000,0.000000,0.030303,0.003584,0.001511,0.000000,0.0000,0.000,...,0.028571,0.020408,0.000000,0.000000,0.000000,0.016667,0.000000,0.003774,0.029412,0.000000
A[C>A]C,0.021739,0.003713,0.045455,0.000000,0.030303,0.000000,0.001511,0.000000,0.0000,0.025,...,0.000000,0.015306,0.064516,0.028571,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A[C>A]G,0.021739,0.000000,0.015152,0.000000,0.000000,0.000000,0.000504,0.000000,0.0000,0.000,...,0.007143,0.005102,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.029412,0.000000
A[C>A]T,0.000000,0.002475,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000,...,0.007143,0.015306,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
A[C>G]A,0.000000,0.001238,0.015152,0.000000,0.000000,0.000000,0.001511,0.018868,0.0625,0.000,...,0.000000,0.005102,0.000000,0.000000,0.000000,0.000000,0.000000,0.003774,0.000000,0.009346
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
T[T>C]T,0.000000,0.000000,0.000000,0.015625,0.030303,0.000000,0.000504,0.000000,0.0000,0.000,...,0.007143,0.000000,0.032258,0.000000,0.000000,0.000000,0.016667,0.000000,0.000000,0.000000
T[T>G]A,0.000000,0.000000,0.015152,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000,...,0.007143,0.005102,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
T[T>G]C,0.000000,0.000000,0.000000,0.000000,0.000000,0.003584,0.000000,0.000000,0.0000,0.000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.016667,0.000000,0.000000,0.000000,0.000000
T[T>G]G,0.000000,0.000000,0.015152,0.000000,0.000000,0.000000,0.000504,0.000000,0.0000,0.000,...,0.000000,0.005102,0.000000,0.000000,0.016129,0.000000,0.000000,0.000000,0.000000,0.000000


In [6]:
tcga_labels

cancer_labels = tcga_labels[tcga_labels['Cancer'] == cancer]
cancer_labels

,Pathway,Mutated_genes,Mode,Cancer,Person
Sample,,,,,
TCGA-A1-A0SB-01A-11D-A142-09,HR,RAD51_het,Heterozygous,BRCA,TCGA-A1-A0SB
TCGA-A1-A0SD-01A-11D-A10Y-09,NER,CUL5_het,Heterozygous,BRCA,TCGA-A1-A0SD
TCGA-A1-A0SD-01A-11D-A10Y-09,BER,APEX2_het,Heterozygous,BRCA,TCGA-A1-A0SD
TCGA-A1-A0SD-01A-11D-A10Y-09,TLS,MAD2L2_het,Heterozygous,BRCA,TCGA-A1-A0SD
TCGA-A1-A0SD-01A-11D-A10Y-09,HR,"BRCA2_het,TOP3A_het,RBBP8_het",Heterozygous,BRCA,TCGA-A1-A0SD
...,...,...,...,...,...
TCGA-GM-A2DL-01A-11D-A18P-09,DS,"ATM_het,RNMT_het,TP53_het",Heterozygous,BRCA,TCGA-GM-A2DL
TCGA-GM-A2DM-01A-11D-A17W-09,TLS,POLQ_het,Heterozygous,BRCA,TCGA-GM-A2DM
TCGA-GM-A2DM-01A-11D-A17W-09,DR,MBD4_het,Heterozygous,BRCA,TCGA-GM-A2DM


In [7]:
# --- Extract Pathway → Gene mapping from all cancers ---
print("Generating pathway × gene matrix...")

# Re-read full label file in case it was filtered earlier
full_labels = tcga_labels

# Initialize mapping
pathway_to_genes = {}

for _, row in full_labels.iterrows():
    pathway = row['Pathway']
    genes = row['Mutated_genes']
    if pd.isna(genes):
        continue

    # Extract base gene names (strip "_het", "_hom", etc.)
    gene_names = [g.split('_')[0] for g in genes.split(',') if '_' in g]
    
    if pathway not in pathway_to_genes:
        pathway_to_genes[pathway] = set()
    pathway_to_genes[pathway].update(gene_names)

# Build binary matrix
all_genes = sorted(set(g for genes in pathway_to_genes.values() for g in genes))
all_pathways = sorted(pathway_to_genes.keys())
binary_matrix = pd.DataFrame(0, index=all_pathways, columns=all_genes)

for pathway, genes in pathway_to_genes.items():
    binary_matrix.loc[pathway, list(genes)] = 1

# Save to CSV
gene_matrix_path = os.path.join(project_root, "data", "processed", "TCGA", "pathway_gene_matrix.csv")
binary_matrix.to_csv(gene_matrix_path)

print(f"  → Saved pathway × gene matrix to {gene_matrix_path}")


Generating pathway × gene matrix...
  → Saved pathway × gene matrix to /Users/sande/Documents/GitHub/SNMF/data/processed/TCGA/pathway_gene_matrix.csv


In [12]:
binary_matrix

,ALKBH2,ALKBH3,APEX1,APEX2,ATM,ATR,ATRIP,BARD1,BLM,BRCA1,...,TOPBP1,TP53,TP53BP1,UBE2T,UNG,XPA,XPC,XRCC2,XRCC5,XRCC6
BER,0,0,1,1,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
DR,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
DS,0,0,0,0,1,1,1,0,0,0,...,1,1,0,0,0,0,0,0,0,0
FA,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
HR,0,0,0,0,0,0,0,1,1,1,...,0,0,1,0,0,0,0,1,0,0
MMR,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
NER,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,0,0,0
NHEJ,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
TLS,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [2]:
# ---- PARSE TCGA (Pathway) LABELS ----

# --- Configuration ---
cancers = ['BRCA', 'COAD', 'UCEC', 'PRAD', 'STAD']
project_root = os.path.join(os.getcwd(), "../..")  # Adjust if needed
tcga_labels_path = os.path.join(project_root, "data", "processed", "TCGA", "annot.txt")
tcga_profiles_dir = os.path.join(project_root, 'data', 'processed', 'TCGA', 'profiles', 'norm_SBS')
output_dir_base = os.path.join(project_root, 'data', 'processed', 'TCGA', 'annotations')

# --- Load labels once ---
tcga_labels = pd.read_csv(tcga_labels_path, sep='\t', header=0, index_col=0)
tcga_labels['Person'] = tcga_labels['Person'].astype(str)
tcga_labels['Pathway'] = tcga_labels['Pathway'].astype(str)

# --- Pathways ---
pathways = ["Control", "MMR", "HR", "BER"]

# --- Loop over cancers ---
for cancer in cancers:
    print(f"Processing {cancer}...")

    # 1. Filter cancer-specific labels
    cancer_labels = tcga_labels[tcga_labels['Cancer'] == cancer]

    # 2. Load TCGA profiles
    tcga_profile_path = os.path.join(tcga_profiles_dir, f"{cancer}.text")
    if not os.path.exists(tcga_profile_path):
        print(f"  → Skipping {cancer}: profile file not found.")
        continue

    tcga_profiles = pd.read_csv(tcga_profile_path, sep='\t', index_col=0)
    tcga_sample_ids = ['-'.join(s.split('-')[:3]) for s in tcga_profiles.columns]

    # 3. Initialize label matrices
    Y_cancer = pd.DataFrame(0, index=pathways, columns=tcga_sample_ids)
    Y_cancer_hom = pd.DataFrame(0, index=pathways, columns=tcga_sample_ids)

    # 4. Populate matrices
    for sid in tcga_sample_ids:
        matches_all = cancer_labels[cancer_labels['Person'] == sid]
        mutated_all = matches_all['Pathway'].unique()
        matches_hom = matches_all[matches_all['Mode'] == 'Total deactivation']
        mutated_hom = matches_hom['Pathway'].unique()

        for p in ['HR', 'MMR', 'BER']:
            if p in mutated_all:
                Y_cancer.loc[p, sid] = 1
            if p in mutated_hom:
                Y_cancer_hom.loc[p, sid] = 1

        if Y_cancer.loc[['HR', 'MMR', 'BER'], sid].sum() == 0:
            Y_cancer.loc['Control', sid] = 1
        if Y_cancer_hom.loc[['HR', 'MMR', 'BER'], sid].sum() == 0:
            Y_cancer_hom.loc['Control', sid] = 1

    # 5. Reorder rows to match desired order
    Y_cancer = Y_cancer.reindex(pathways)
    Y_cancer_hom = Y_cancer_hom.reindex(pathways)

    # 6. Save
    output_path = os.path.join(output_dir_base, cancer)
    os.makedirs(output_path, exist_ok=True)
    Y_cancer.to_csv(os.path.join(output_path, 'Y_cancer.text'), sep='\t')
    Y_cancer_hom.to_csv(os.path.join(output_path, 'Y_cancer_hom.text'), sep='\t')

    print(f"  → Saved Y_cancer and Y_cancer_hom for {cancer} to {output_path}")


Processing BRCA...
  → Saved Y_cancer and Y_cancer_hom for BRCA to /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/BRCA
Processing COAD...
  → Saved Y_cancer and Y_cancer_hom for COAD to /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/COAD
Processing UCEC...
  → Saved Y_cancer and Y_cancer_hom for UCEC to /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/UCEC
Processing PRAD...
  → Saved Y_cancer and Y_cancer_hom for PRAD to /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/PRAD
Processing STAD...
  → Saved Y_cancer and Y_cancer_hom for STAD to /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/STAD


In [3]:
Y_cancer

,TCGA-3M-AB46,TCGA-3M-AB47,TCGA-B7-5816,TCGA-B7-5818,TCGA-B7-A5TI,TCGA-B7-A5TJ,TCGA-B7-A5TK,TCGA-B7-A5TN,TCGA-BR-4183,TCGA-BR-4184,...,TCGA-VQ-AA6A,TCGA-VQ-AA6B,TCGA-VQ-AA6D,TCGA-VQ-AA6F,TCGA-VQ-AA6G,TCGA-VQ-AA6I,TCGA-VQ-AA6J,TCGA-VQ-AA6K,TCGA-ZA-A8F6,TCGA-ZQ-A9CR
Control,0,1,0,0,0,0,0,0,1,1,...,1,1,0,1,0,1,0,0,0,1
MMR,1,0,1,0,1,1,0,0,0,0,...,0,0,1,0,1,0,1,1,1,0
HR,1,0,0,1,1,0,1,1,0,0,...,0,0,1,0,1,0,1,1,0,0
BER,1,0,0,1,1,1,0,0,0,0,...,0,0,0,0,1,0,1,1,0,0


In [22]:
tcga_idx = "TCGA-A2-A0YC"
cancer_labels[cancer_labels['Person'] == tcga_idx]

 



,Pathway,Mutated_genes,Mode,Cancer,Person
Sample,,,,,
TCGA-A2-A0YC-01A-11D-A117-09,NER,CUL5_het,Heterozygous,BRCA,TCGA-A2-A0YC
TCGA-A2-A0YC-01A-11D-A117-09,FA,FANCA_het,Heterozygous,BRCA,TCGA-A2-A0YC
TCGA-A2-A0YC-01A-11D-A117-09,DS,"ATM_het,CHEK1_het,TP53_het",Heterozygous,BRCA,TCGA-A2-A0YC


In [ ]:
## ==== PARSE TCGA GENE LABELS ===
from collections import defaultdict

# --- Configuration ---
cancers = ['BRCA', 'COAD', 'UCEC', 'PRAD', 'STAD']
project_root = os.path.join(os.getcwd(), "../..")  # or set this explicitly
tcga_labels_path = os.path.join(project_root, "data", "processed", "TCGA", "annot.txt")
tcga_profiles_dir = os.path.join(project_root, 'data', 'processed', 'TCGA', 'profiles', 'norm_SBS')
output_dir_base = os.path.join(project_root, 'data', 'processed', 'TCGA', 'annotations')

# --- Load labels once ---
tcga_labels = pd.read_csv(tcga_labels_path, sep='\t', header=0, index_col=0)
tcga_labels['Person'] = tcga_labels['Person'].astype(str)
tcga_labels['Pathway'] = tcga_labels['Pathway'].astype(str)


# --- Loop per cancer ---
for cancer in cancers:
    print(f"Processing {cancer} gene matrices...")

    # Load sample list from TCGA profile file
    tcga_profile_path = os.path.join(tcga_profiles_dir, f"{cancer}.text")
    if not os.path.exists(tcga_profile_path):
        print(f"  → Skipping {cancer}: profile file not found.")
        continue

    tcga_profiles = pd.read_csv(tcga_profile_path, sep='\t', index_col=0)
    tcga_sample_ids = ['-'.join(s.split('-')[:3]) for s in tcga_profiles.columns]

    # Filter cancer-specific labels
    cancer_labels = tcga_labels[tcga_labels['Cancer'] == cancer]

    # Build gene sets and patient-gene mapping
    gene_matrix_het = defaultdict(set)
    gene_matrix_hom = defaultdict(set)
    all_genes_het = set()
    all_genes_hom = set()

    for _, row in cancer_labels.iterrows():
        sid = row['Person']
        if sid not in tcga_sample_ids:
            continue  # skip patients not in the profile

        for g in row['Mutated_genes'].split(','):
            if not g: continue
            if g.endswith('_het'):
                gene = g[:-4]
                gene_matrix_het[sid].add(gene)
                all_genes_het.add(gene)
            elif g.endswith('_hom'):
                gene = g[:-4]
                gene_matrix_hom[sid].add(gene)
                all_genes_hom.add(gene)

    # Make sorted gene lists
    genes_het = sorted(all_genes_het)
    genes_hom = sorted(all_genes_hom)

    # Initialize matrices with zeros, rows in tcga_sample_ids order
    het_matrix = pd.DataFrame(0, index=tcga_sample_ids, columns=genes_het)
    hom_matrix = pd.DataFrame(0, index=tcga_sample_ids, columns=genes_hom)

    # Fill in matrices
    for sid in tcga_sample_ids:
        for gene in gene_matrix_het.get(sid, []):
            het_matrix.loc[sid, gene] = 1
        for gene in gene_matrix_hom.get(sid, []):
            hom_matrix.loc[sid, gene] = 1

    # Save as .text files
    out_dir = os.path.join(project_root, "data", "processed", "TCGA", "annotations", cancer)
    os.makedirs(out_dir, exist_ok=True)
    het_matrix.to_csv(os.path.join(out_dir, "gene_matrix_het.text"), sep='\t')
    hom_matrix.to_csv(os.path.join(outdd_dir, "gene_matrix_hom.text"), sep='\t')
    print(f"  → Saved het and hom matrices to: {out_dir}")


Processing BRCA gene matrices...
  → Saved het and hom matrices to: /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/BRCA
Processing COAD gene matrices...
  → Saved het and hom matrices to: /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/COAD
Processing UCEC gene matrices...
  → Saved het and hom matrices to: /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/UCEC
Processing PRAD gene matrices...
  → Saved het and hom matrices to: /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/PRAD
Processing STAD gene matrices...
  → Saved het and hom matrices to: /Users/sande/Documents/GitHub/SNMF/notebooks/reproduce/../../data/processed/TCGA/annotations/STAD


In [41]:
hom_matrix.sum(axis=0).sort_values(ascending=False)

TP53      25
MLH1       6
ERCC2      2
TOP3A      2
ALKBH3     1
BRCA1      1
EXO1       1
FANCA      1
FANCB      1
MSH3       1
POLK       1
POLL       1
POLN       1
UBE2T      1
dtype: int64

In [24]:
from itertools import combinations
from sklearn.metrics import jaccard_score

# --- 1. Basic counts ---
row_sums = Y_cancer.sum(axis=1)

# --- 2. Compute total active pathways per sample ---
sample_activity = Y_cancer.sum(axis=0)  # sums across pathways → gives 1 (single), >1 (multi)

# --- 3. For each pathway, compute exclusive (only) and co-occurring (multi) counts ---
only_counts = {}
multi_counts = {}

for pathway in Y_cancer.index:
    active = Y_cancer.loc[pathway] == 1
    only_counts[pathway] = ((active) & (sample_activity == 1)).sum()
    multi_counts[pathway] = ((active) & (sample_activity > 1)).sum()

# --- 4. Create final summary DataFrame ---
summary = pd.DataFrame({
    'Total': row_sums,
    'Only': pd.Series(only_counts),
    'Multi': pd.Series(multi_counts)
})

print(summary)

# --- 2. Compute pairwise Jaccard similarity between pathways ---
# Convert to binary arrays
binary_matrix = Y_cancer.values.astype(bool)

# Store results in a DataFrame
jaccard_df = pd.DataFrame(index=Y_cancer.index, columns=Y_cancer.index, dtype=float)

# Compute Jaccard similarity for each pair
for p1, p2 in combinations(Y_cancer.index, 2):
    jaccard = jaccard_score(binary_matrix[Y_cancer.index.get_loc(p1)],
                            binary_matrix[Y_cancer.index.get_loc(p2)])
    jaccard_df.loc[p1, p2] = jaccard
    jaccard_df.loc[p2, p1] = jaccard

# Fill diagonal with 1.0
np.fill_diagonal(jaccard_df.values, 1.0)

print("\nJaccard Similarity Matrix:\n", jaccard_df)


         Total  Only  Multi
Control    490   490      0
HR         246    93    153
MMR        139    15    124
BER        163    34    129

Jaccard Similarity Matrix:
          Control        HR       MMR       BER
Control      1.0  0.000000  0.000000  0.000000
HR           0.0  1.000000  0.441948  0.430070
MMR          0.0  0.441948  1.000000  0.451923
BER          0.0  0.430070  0.451923  1.000000


In [16]:
cancer = 'BRCA'

os.path.join(project_root, "data", "processed", "TCGA", "processed", "BRCA", "TCGA_genes_BRCA.pkl")


TCGA_samples = pd.read_pickle(os.path.join(project_root, "data", "processed", "TCGA", "processed", "BRCA", "TCGA_samples_BRCA.pkl"))
TCGA_genes = pd.read_pickle(os.path.join(project_root, "data", "processed", "TCGA", "processed", "BRCA", "TCGA_genes_BRCA.pkl"))

TCGA_samples

,A[C>A]A,A[C>A]C,A[C>A]G,A[C>A]T,A[C>G]A,A[C>G]C,A[C>G]G,A[C>G]T,A[C>T]A,A[C>T]C,...,REV1_mut,RNMT_mut,EME1_mut,PMS2_mut,ATM_mut,FANCC_mut,FANCB_mut,MSH3_mut,ATR_mut,APEX2_mut
TCGA-3C-AAAU,0.0,0.021739,0.021739,0.0,0.0,0.0,0.0,0.065217,0.0,0.021739,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
TCGA-3C-AALI,0.003713,0.003713,0.0,0.002475,0.001238,0.001238,0.001238,0.002475,0.003713,0.002475,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
TCGA-3C-AALJ,0.0,0.045455,0.015152,0.0,0.015152,0.0,0.0,0.0,0.0,0.045455,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
TCGA-3C-AALK,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.015625,0.015625,0.0,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
TCGA-4H-AAAK,0.030303,0.030303,0.0,0.0,0.0,0.0,0.0,0.0,0.030303,0.0,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TCGA-WT-AB44,0.016667,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.016667,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
TCGA-XX-A899,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.033333,0.0,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
TCGA-XX-A89A,0.003774,0.0,0.0,0.0,0.003774,0.0,0.0,0.003774,0.003774,0.0,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT
TCGA-Z7-A8R5,0.029412,0.0,0.029412,0.0,0.0,0.029412,0.0,0.0,0.0,0.0,...,WT,WT,WT,WT,WT,WT,WT,WT,WT,WT


In [ ]:
# Create Y Matrix - pathway deficiency 

## Get matching TCGA samples

## Get gene KOs



